# v3 article extraction — scored against the v3 gold article list

The frozen v3 pipeline (tag `extraction-v3`) on the **held-out v3 test volumes** (Penny Magazine 1838, Saturday
Magazine 1834, Chambers's Edinburgh Journal 1843 — three issues each, never used to write rules) scored against
`gold_article_list_test_v3.csv`, which Owen annotated from the printed pages alone, without seeing the output
(guide: `documentation/article_extraction_pipeline_iterative_build/gold_article_list_annotation_guide.md`).
The gold list was corrected afterwards from the printed pages only; the score before corrections is in commit
`3fc1815`. The output read is `output/article_extraction/v3/`, frozen. Page numbers are PDF pages; the check
images are in `output/article_extraction/v3/<volume>/reading_order/`.

**How items are matched.** Each gold item is located in the output by its first and last words
(OCR-tolerant matching on the item's own pages). Every number below is computed by `score_gold.py`; nothing is
judged by hand.

| Outcome | Meaning |
|---|---|
| **exact** | the item opens a pipeline article of its own, and its last words are in that article |
| **merged** | its pipeline article also holds the start of the gold item before or after it (a missed boundary) |
| **split** | its first and last words land in different pipeline articles (a false boundary inside it) |
| **lost** | its first words are in no article's text (set aside as furniture, absorbed into other matter, or not found) |

- **Boundary recall** — gold items that open their own pipeline article. **Boundary precision** — pipeline
  articles that hold a gold item's start; the rest are *extra* pieces (fragments of split items, masthead lines…).
- **Fields** are scored for items that open their own article, across all the pipeline articles the item runs
  through if it was split.

Set `VERSION` below to score another version against the same gold list.

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

HERE = next(p for p in [Path("."), Path("evaluation/evaluating_v3_article_extraction")]
            if (p / "gold_article_list_test_v3.csv").exists())
SCORER_DIR = HERE.resolve().parent / "evaluating_v2_article_extraction"  # score_gold.py lives there
sys.path.insert(0, str(SCORER_DIR))
from score_gold import load_gold, load_pipeline, score, summary_tables, write, VOLUME_ALIASES

VERSION = "v3"
GOLD = HERE / "gold_article_list_test_v3.csv"
OUT_DIR = HERE  # results: <VERSION>_gold_comp_eval_*.csv / _summary.md

## Run the scoring and save the results

In [2]:
gold = load_gold(GOLD)
res = score(VERSION, gold)
summary_path = write(res, VERSION, OUT_DIR, GOLD)
items, arts, pieces, imprints = res["items"], res["pipeline_articles"], res["pieces"], res["imprints"]
tables = summary_tables(res)
own = items[items.opens_own_article]
print(f"Gold list: {len(gold)} rows — {gold.article_type.value_counts().to_dict()}")
print(f"Scored {len(items)} gold items against {len(arts)} pipeline articles ({VERSION})")
print(f"Wrote {OUT_DIR}/{VERSION}_gold_comp_eval_*.csv and {summary_path.name}")

Gold list: 85 rows — {'article': 75, 'imprint': 9, 'editorial_notice': 1}
Scored 76 gold items against 98 pipeline articles (v3)
Wrote ./v3_gold_comp_eval_*.csv and v3_gold_comp_eval_summary.md


## 1. Headline: items and boundaries

In [3]:
tables['items']

,chambers_1843,saturday_1834,penny_1838,all
gold items,39,30,7,76
pipeline articles,48,39,11,98
exact,32/39 (82%),26/30 (87%),4/7 (57%),62/76 (82%)
merged,2/39 (5%),0/30 (0%),0/7 (0%),2/76 (3%)
split,5/39 (13%),4/30 (13%),2/7 (29%),11/76 (14%)
lost,0/39 (0%),0/30 (0%),1/7 (14%),1/76 (1%)
boundary recall,38/39 (97%),30/30 (100%),6/7 (86%),74/76 (97%)
boundary precision,38/48 (79%),29/39 (74%),6/11 (55%),73/98 (74%)
extra pipeline articles,10,10,5,25


In [4]:
# Item outcome counts per volume
pd.crosstab(items.volume, items.status, margins=True, margins_name="total")

status,exact,lost,merged,split,total
volume,,,,,
chambers_1843,32,0,2,5,39
penny_1838,4,1,0,2,7
saturday_1834,26,0,0,4,30
total,62,1,2,11,76


## 2. Successes — items extracted exactly

In [5]:
exact = items[items.status == "exact"]
print(f"{len(exact)} of {len(items)} gold items extracted exactly")
exact[["volume", "gold_id", "gold_title", "pipeline_title", "start_article_id"]]

62 of 76 gold items extracted exactly


,volume,gold_id,gold_title,pipeline_title,start_article_id
0,chambers_1843,1,Tact.,,chambers_1843_0001
3,chambers_1843,4,Scotch Banks.,SCOTCH BANKS,chambers_1843_0010
4,chambers_1843,5,A Word on Mechanics’ Institutions Generally.,A WORD ON MECHANICS' GENERALLY,chambers_1843_0011
6,chambers_1843,7,A Peevish Old Scotswoman.,A PEEVISH OLD SCOTSWOMAN,chambers_1843_0014
8,chambers_1843,9,Prejudice as to Food.,PREJUDICE AS TO FOOD,chambers_1843_0018
9,chambers_1843,10,Bleeding at the Nose.,BLEEDING AT THE NOSE,chambers_1843_0019
10,chambers_1843,11,The French Press.,THE FRENCH PRESS,chambers_1843_0020
11,chambers_1843,12,An Indian’s Ingenuity.,AN INDIAN'S INGENUITY,chambers_1843_0021
12,chambers_1843,13,"Hear, Hear!","HEAR, HEAR!",chambers_1843_0022
13,chambers_1843,15,Dinners.,DINNERS,chambers_1843_0023


## 3. Errors — merged items (missed boundaries)

In [6]:
items[items.status.str.startswith("merged")][
    ["volume", "gold_id", "start_pdf_page", "gold_title", "merged_with", "start_article_id", "pipeline_title"]]

,volume,gold_id,start_pdf_page,gold_title,merged_with,start_article_id,pipeline_title
31,chambers_1843,34,28,Agriculture in England.,35,chambers_1843_0042,AGRICULTURE IN ENGLAND
32,chambers_1843,35,29,The Walhalla.,34,chambers_1843_0042,


## 4. Errors — split items (false boundaries inside an item)

In [7]:
split = items[items.status.str.contains("split")]
split[["volume", "gold_id", "start_pdf_page", "gold_title", "start_article_id", "end_article_id", "pipeline_articles"]]

,volume,gold_id,start_pdf_page,gold_title,start_article_id,end_article_id,pipeline_articles
1,chambers_1843,2,8,“Rural and Domestic Life In Germany.”,chambers_1843_0002,chambers_1843_0005,chambers_1843_0002;chambers_1843_0003;chambers_1843_0004;chambers_...
2,chambers_1843,3,9,A Wedding in Garrison.*,chambers_1843_0006,chambers_1843_0009,chambers_1843_0006;chambers_1843_0007;chambers_1843_0008;chambers_...
5,chambers_1843,6,12,Nursery Rhymes of England.,chambers_1843_0012,chambers_1843_0013,chambers_1843_0012;chambers_1843_0013
7,chambers_1843,8,13,Genealogy of the Queen.,chambers_1843_0015,chambers_1843_0017,chambers_1843_0015;chambers_1843_0016;chambers_1843_0017
24,chambers_1843,27,23,Ensign Maryn’s Frist Scrape.,chambers_1843_0034,chambers_1843_0035,chambers_1843_0034;chambers_1843_0035
48,saturday_1834,52,10,On Wills. No. IV Wills of Personal Property Continued.,saturday_1834_0007,saturday_1834_0012,saturday_1834_0007;saturday_1834_0008;saturday_1834_0009;saturday_...
49,saturday_1834,53,11,Address to A Godchild.,saturday_1834_0013,saturday_1834_0015,saturday_1834_0013;saturday_1834_0014;saturday_1834_0015
56,saturday_1834,61,14,Sketches of the Highlands and Islands of Scotland.,saturday_1834_0022,saturday_1834_0026,saturday_1834_0022;saturday_1834_0023;saturday_1834_0024;saturday_...
62,saturday_1834,68,25,Henry Prices of Wales; Eldest Son of King James The First.,saturday_1834_0032,saturday_1834_0033,saturday_1834_0032;saturday_1834_0033
70,penny_1838,77,9,Christmas Games in the Olden Time.,penny_1838_0002,penny_1838_0004,penny_1838_0002;penny_1838_0003;penny_1838_0004


## 5. Errors — lost items (opening words not in any article's text)

In [8]:
items[items.status == "lost"][["volume", "gold_id", "start_pdf_page", "gold_title", "start_found_in", "start_score"]]

,volume,gold_id,start_pdf_page,gold_title,start_found_in,start_score
75,penny_1838,84,25,,box role: furniture,1.0


## 6. Extra pipeline articles (hold no gold item's start)

Usually fragments of split items (`gold_ends` shows a gold item whose *last* words fall in it) or page
furniture taken as an article (masthead lines, prices, dates).

In [9]:
extra = arts[arts.status == "extra"]
print(extra.volume.value_counts().to_dict())
extra[["volume", "article_id", "pipeline_title", "pages", "word_count", "gold_ends", "text_start"]]

{'chambers_1843': 10, 'saturday_1834': 10, 'penny_1838': 5}


,volume,article_id,pipeline_title,pages,word_count,gold_ends,text_start
2,chambers_1843,chambers_1843_0003,GERMAN ECONOMY,8,603,,"""Nothing that can possibly be made use of is lost. The children"
3,chambers_1843,chambers_1843_0004,THE GERMAN PEASANT'S HOME,8,388,,"""The English of the working-class can indeed form no conception of..."
4,chambers_1843,chambers_1843_0005,GERMAN DOMESTIC LIFE,8;9,1336,2,"""Early hours and simple living distinguish the Germans. Three meal..."
6,chambers_1843,chambers_1843_0007,SERVANTS,9,149,,Sometimes it is-and I have had a Benjamin's portion of such blessi...
7,chambers_1843,chambers_1843_0008,NaN,9,25,,"was of both at G-, both during and after tsiege; yet my"
8,chambers_1843,chambers_1843_0009,NaN,9;10,3644,3,perhaps because we had not then 'supped with horrors; perhaps beca...
12,chambers_1843,chambers_1843_0013,NaN,13,482,6,"The king sent his lady on the thirteenth Yule day, Three stalks"
15,chambers_1843,chambers_1843_0016,1. ANGLO-SAXON DESCENT OF HER MAJESTY,13,360,,"A.D. 800. Egbert, founder of the Anglo-Saxon monarchy. (1.) 836. E..."
16,chambers_1843,chambers_1843_0017,2. SCOTTISH DESCENT OF HER MAJESTY,13;14,2489,8,"A. D. 843. Kenneth MacAlpin, styled Kenneth II. (1.) 863. Constant..."
34,chambers_1843,chambers_1843_0035,WILLIAM MARTYN,23;24;25,4221,27,"Mrs William Martin, Stockwell Street, Glasgow."" This rhapsody I ca..."


## 7. Fields — summary

In [10]:
tables['fields']

,chambers_1843,saturday_1834,penny_1838,all
items scored (open their own article),38,30,6,74
title,37/38 (97%),25/30 (83%),3/6 (50%),65/74 (88%)
front matter pieces in place,6/6 (100%),0/3 (0%),2/3 (67%),8/12 (67%)
back matter pieces in place,23/30 (77%),23/34 (68%),5/10 (50%),51/74 (69%)
series (where either has one),2/3 (67%),0/5 (0%),0/2 (0%),2/10 (20%)
continued from,38/38 (100%),28/30 (93%),6/6 (100%),72/74 (97%)
to be continued,38/38 (100%),30/30 (100%),6/6 (100%),74/74 (100%)
editorial notice / article,38/38 (100%),30/30 (100%),6/6 (100%),74/74 (100%)
illustration count,38/38 (100%),26/30 (87%),5/6 (83%),69/74 (93%)
issue number,38/38 (100%),30/30 (100%),6/6 (100%),74/74 (100%)


### 7a. Titles that disagree

In [11]:
own[~own.title_result.isin(["correct", "both untitled"])][
    ["volume", "gold_id", "start_pdf_page", "gold_title", "pipeline_title", "title_result"]]

,volume,gold_id,start_pdf_page,gold_title,pipeline_title,title_result
0,chambers_1843,1,7,Tact.,,missing
44,saturday_1834,48,8,No. II. On the Discovery of Mineral Veins and the Mode of First Op...,THE MINES OF GREAT BRITAIN,wrong
48,saturday_1834,52,10,On Wills. No. IV Wills of Personal Property Continued.,THE MINES OF GREAT BRITAIN,wrong
56,saturday_1834,61,14,Sketches of the Highlands and Islands of Scotland.,PART THE SECOND. IONA; INSULAR CHURCHES,wrong
57,saturday_1834,63,22,The Forbidden Fruit.,"OR, EVE'S APPLE TREE OF CEYLON",wrong
67,saturday_1834,73,27,No. IV. Heat. Temperature. Radiation. Conduction.,FAMILIAR ILLUSTRATIONS OF EXPERI- MENTAL SCIENCE,wrong
69,penny_1838,76,6,No. V. Little Musgrave. - The Friar of Orders Gray,ENGLISH ROMANTIC BALLADS,wrong
72,penny_1838,80,14,Le Comte Alexandre-Louis-Joseph De Laborde.,PUBLISHED EVERY SATURDAY,wrong
74,penny_1838,83,18,"Public Improvements. - No. II., Provincial Towns.",,missing


### 7b. Front and back matter pieces

Each gold piece is found **in place** (the pipeline's front/back matter), **in text** (left in the article text),
elsewhere, or **missing** (set aside or not found). Footnotes — many in Chambers — are counted separately; some
give sources, others explain a word or concept.

In [12]:
tables['pieces']

result,in place,in text,in title,missing,total
kind,,,,,
back matter,23,9,0,5,37
back matter (footnote),28,9,0,0,37
front matter,8,1,3,0,12
total,59,19,3,5,86


In [13]:
# Pieces not in place
pieces[pieces.result != "in place"][["volume", "gold_id", "field", "footnote", "result", "piece"]]

,volume,gold_id,field,footnote,result,piece
4,chambers_1843,5,back_matter,True,in text,* These statements are partly derived from “ A Report of the state...
9,chambers_1843,12,back_matter,False,in text,Newspaper paragraph.
22,chambers_1843,25,back_matter,False,in text,Scottish Jests and Anecdotes.
31,chambers_1843,37,back_matter,False,in text,Binns on Sleep.
33,chambers_1843,39,back_matter,False,in text,Bentley’s Miscellany.
34,chambers_1843,40,back_matter,False,missing,"Man the Reformer, by R. W. Emerson."
35,chambers_1843,41,back_matter,False,in text,Naval and Military Gazette.
37,saturday_1834,43,back_matter,True,in text,"† The water of St. Ethelbert’s well, not far from the Cathedral, i..."
38,saturday_1834,43,back_matter,False,missing,∆.
42,saturday_1834,48,back_matter,True,in text,"* See Vol. III., p. 178, and Vol. IV. p. 43. p. 43"


In [14]:
# Pipeline front/back matter with no gold counterpart (extra pieces), per item
x = own[(own.front_matter_extra > 0) | (own.back_matter_extra > 0)]
x[["volume", "gold_id", "gold_title", "front_matter_extra", "front_matter_pipeline",
   "back_matter_extra", "back_matter_pipeline"]]

,volume,gold_id,gold_title,front_matter_extra,front_matter_pipeline,back_matter_extra,back_matter_pipeline
5,chambers_1843,6,Nursery Rhymes of England.,0.0,,1.0,"* Popular Rhymes, Fireside Stories, and Amusements of Scotland. Co..."
31,chambers_1843,34,Agriculture in England.,0.0,,1.0,"* See article, ""An Englishman's Tour among Scotch Farmers,"" No. 50..."
48,saturday_1834,52,On Wills. No. IV Wills of Personal Property Continued.,0.0,,3.0,Life and Times of Henry the Eighth | § 6. On the Form and Language...
73,penny_1838,81,Quacks and Quack Medicines.,0.0,[From the 'Working-Man's Companion' for 1839.] [Concluded from No....,1.0,"* According to Gray's 'Supplement to the Pharmacopœia,' Barclay's ..."


### 7c. Series and continuations that disagree

In [15]:
g = gold.set_index(["volume", "gold_id"])
s = own[~own.series_result.isin(["correct", "none (both)"])].copy()
s["gold_series"] = [g.loc[(v, i), "series"] for v, i in zip(s.volume, s.gold_id)]
s[["volume", "gold_id", "gold_title", "gold_series", "pipeline_series", "series_result"]]

,volume,gold_id,gold_title,gold_series,pipeline_series,series_result
1,chambers_1843,2,“Rural and Domestic Life In Germany.”,,Rural and Domestic Life,extra
44,saturday_1834,48,No. II. On the Discovery of Mineral Veins and the Mode of First Op...,The Mines of Great Britain.,,missing
48,saturday_1834,52,On Wills. No. IV Wills of Personal Property Continued.,On Wills.,,missing
56,saturday_1834,61,Sketches of the Highlands and Islands of Scotland.,,Sketches of the Highlands and Islands of Scotland,extra
57,saturday_1834,63,The Forbidden Fruit.,,The Forbidden Fruit,extra
67,saturday_1834,73,No. IV. Heat. Temperature. Radiation. Conduction.,Familiar Illustrations of Experimental Science.,,missing
69,penny_1838,76,No. V. Little Musgrave. - The Friar of Orders Gray,English Romantic Ballads.,,missing
74,penny_1838,83,"Public Improvements. - No. II., Provincial Towns.",Public Improvements.,,missing


In [16]:
own[~own.continued_from_result.str.startswith("correct") | ~own.to_be_continued_result.str.startswith("correct")][
    ["volume", "gold_id", "gold_title", "continued_from_result", "to_be_continued_result"]]

,volume,gold_id,gold_title,continued_from_result,to_be_continued_result
48,saturday_1834,52,On Wills. No. IV Wills of Personal Property Continued.,missing,correct (yes)
56,saturday_1834,61,Sketches of the Highlands and Islands of Scotland.,missing,correct (no)


### 7d. Illustrations and article type that disagree

In [17]:
own[own.figures_result != "correct"][["volume", "gold_id", "gold_title", "figures_gold", "figures_pipeline"]]

,volume,gold_id,gold_title,figures_gold,figures_pipeline
39,saturday_1834,43,Hereford Cathedral.,1.0,0.0
48,saturday_1834,52,On Wills. No. IV Wills of Personal Property Continued.,0.0,2.0
51,saturday_1834,55,On the Structure of Animals. II.,1.0,2.0
62,saturday_1834,68,Henry Prices of Wales; Eldest Son of King James The First.,0.0,1.0
74,penny_1838,83,"Public Improvements. - No. II., Provincial Towns.",7.0,8.0


In [18]:
own[~own.type_result.str.startswith("correct")][["volume", "gold_id", "gold_title", "article_type", "type_result"]]

,volume,gold_id,gold_title,article_type,type_result


## 8. Issue metadata — number and date

In [19]:
issue = own.groupby(["volume", "issue_no"]).agg(
    pipeline_issue_no=("pipeline_issue_no", lambda x: ", ".join(sorted(set(map(str, x))))),
    pipeline_issue_date=("pipeline_issue_date", lambda x: ", ".join(sorted(set(map(str, x))))),
    number_correct=("issue_no_result", lambda x: (x == "correct").mean().round(2)),
    date_correct=("issue_date_result", lambda x: (x == "correct").mean().round(2)))
issue

pipeline_issue_no pipeline_issue_date  number_correct  date_correct
volume        issue_no                                                                    
chambers_1843 573                    573    January 21, 1843             1.0           1.0
              574                    574    January 28, 1843             1.0           1.0
              575                    575    February 4, 1843             1.0           1.0
penny_1838    431                    431   December 22, 1838             1.0           1.0
              432                    432   December 29, 1838             1.0           1.0
              433                    433   December 31, 1838             1.0           1.0
saturday_1834 138                    138     August 30, 1834             1.0           1.0
              139                    139                                 1.0           0.0
              140                    140   September 6, 1834             1.0           1.0

## 9. Imprints (publishing history)

In [20]:
if "imprints" in tables:
    display(tables["imprints"])
imprints[["volume", "gold_id", "issue_no", "result", "missing_part_found_in", "pipeline_imprint"]]

result,complete,missing,total
volume,,,
chambers_1843,3,0,3
penny_1838,3,0,3
saturday_1834,2,1,3
total,8,1,9


,volume,gold_id,issue_no,result,missing_part_found_in,pipeline_imprint
0,chambers_1843,14,573,complete,,"LONDON: Published, with permission of the proprietors, by W. S. OR..."
1,chambers_1843,26,574,complete,,"LONDON: Published, with permission of the proprietors, by W. S. OR..."
2,chambers_1843,42,575,complete,,"LONDON: Published, with permission of the proprietors, by W. S. OR..."
3,saturday_1834,60,138,complete,,"LONDON: JOHN WILLIAM PARKER, WEST STRAND. PUBLISHED IN WEEKLY NUMB..."
4,saturday_1834,62,139,complete,,"LONDON: Published by JOHN WILLIAM PARKER, WEST STRAND; and sold by..."
5,saturday_1834,75,140,missing,not found,
6,penny_1838,79,431,complete,,The Office of the Society for the Diffusion of Useful Knowledge is...
7,penny_1838,82,432,complete,,The Office of the Society for the Diffusion of Useful Knowledge is...
8,penny_1838,85,433,complete,,The Office of the Society for the Diffusion of Useful Knowledge is...


## 10. Look up one gold item

In [21]:
def show(volume, gold_id):
    """A gold row next to what the pipeline produced for it."""
    row = gold[(gold.volume == volume) & (gold.gold_id == gold_id)].iloc[0]
    it = items[(items.volume == volume) & (items.gold_id == gold_id)]
    print("GOLD:", {k: v for k, v in row.items() if v and k not in ("pipeline_volume",)})
    if len(it):
        it = it.iloc[0]
        print("\nRESULT:", it.status, "| title:", it.get("title_result", ""), "| merged with:", it.merged_with or "-")
        pipe = load_pipeline(VERSION, VOLUME_ALIASES.get(volume, volume))["articles"]
        ids = str(it.get("pipeline_articles") or it.start_article_id or "").split(";")
        for aid in [a for a in ids if a in pipe.index]:
            a = pipe.loc[aid]
            print(f"\n{aid}: {a.title!r}  pages {a.pages}")
            print("  front:", a.front_matter if isinstance(a.front_matter, str) else "-")
            print("  text :", " ".join(str(a.text).split()[:30]), "…")
            print("  back :", a.back_matter if isinstance(a.back_matter, str) else "-")

show("saturday_1834", 61)

GOLD: {'gold_id': 61, 'volume': 'saturday_1834', 'issue_no': '139', 'issue_date': '8.1834', 'start_pdf_page': 14, 'start_printed_page': '82', 'start_column': 'left', 'end_pdf_page': '20', 'article_type': 'article', 'title': 'Sketches of the Highlands and Islands of Scotland.', 'first_words': 'Early on the 25th,', 'last_words': 'cheaper rate than peat.', 'front_matter': 'Part the Second. | Iona; Insular Churches.', 'back_matter': '* The practice alluded to, reminds us of the superstition of the American Indian, described by the poet: “Who thinks, admitted to an equal say, His faithful dog shall bear him company.” | He is now called sheriff. | * The ecclesiastical jurisdiction in Scotland is vested in the presbytery, composed of the ministers and some of the elders of a few adjoining parishes: in a synod, which comprises several presbyteries, and is composed of ministers and elders, elected by the presbyteries: and of the General Assmbly, similar constituted, in which the ultimate appell